##### After obtaining your API_KEY and API_SECRET from E-Trade, 
##### you can use the following template to execute your own real-time strategy

In [ ]:
#Set Key and Secret First
#It could be either sandbox or production

CONSUMER_KEY=""
CONSUMER_SECRET=""
SANDBOX_BASE_URL="https://apisb.etrade.com"
PROD_BASE_URL="https://api.etrade.com"

#Set the base url to either SANDBOX_BASE_URL or PROD_BASE_URL
base_url=SANDBOX_BASE_URL #Change to PROD_BASE_URL for production

In [ ]:
# Import necessary libraries

import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import webbrowser
import logging
from logging.handlers import RotatingFileHandler
from rauth import OAuth1Service
import random
import time
import threading

In [ ]:
# Set up logging configuration
logger = logging.getLogger('my_logger')
logger.setLevel(logging.DEBUG)
handler = RotatingFileHandler("python_client.log", maxBytes=5*1024*1024, backupCount=3)
FORMAT = "%(asctime)-15s %(message)s"
fmt = logging.Formatter(FORMAT, datefmt='%m/%d/%Y %I:%M:%S %p')
handler.setFormatter(fmt)
logger.addHandler(handler)

headers = {"Content-Type": "application/xml", "consumerKey": CONSUMER_KEY}

etrade = OAuth1Service(
    name="etrade",
    consumer_key=CONSUMER_KEY,
    consumer_secret=CONSUMER_SECRET,
    request_token_url="https://api.etrade.com/oauth/request_token",
    access_token_url="https://api.etrade.com/oauth/access_token",
    authorize_url="https://us.etrade.com/e/t/etws/authorize?key={}&token={}",
    base_url="https://api.etrade.com"
)

# Get the request token
request_token, request_token_secret = etrade.get_request_token(
    params={"oauth_callback": "oob", "format": "json"}
)

# Open the browser to the authorize URL to get the verifier code
authorize_url = etrade.authorize_url.format(etrade.consumer_key, request_token)
webbrowser.open(authorize_url)

##### A browser window will automatically open to authorize access. After you log in, you will receive a verifier code. Assign this code to the text_code variable to authorize your session and gain access to your account."

In [ ]:
# Get the access token with the verifier code
text_code = ""
session = etrade.get_auth_session(request_token, request_token_secret, params={"oauth_verifier": text_code})

# Print session status
print(session)

In [ ]:
# Get the account list
account_list = session.get(base_url + "/v1/accounts/list.json")
account_list = pd.DataFrame(account_list.json()['AccountListResponse']['Accounts']['Account'])
account_list

#### Select the account you wish to trade with, obtain the accountIdKey, and assign it into below

In [ ]:
# Choose account to trade with and assign accountIdKey
accountIdKey = ""
previewurl = base_url + "/v1/accounts/" + accountIdKey + "/orders/preview.json"
orderurl = base_url + "/v1/accounts/" + accountIdKey + "/orders/place.json

#### Define function to fetch data, make predictions, execute trades, and close positions.

In [ ]:
def quant(stock, amount, threshold, interval):

    #get current stock quote and Price
    quoteurl=f"{base_url}/v1/market/quote/{stock}.json"
    response=session.get(quoteurl, header_auth=True)
    price=response.json()['QuoteResponse']['QuoteData'][0]['All']['lastTrade']


    #get today's date
    date = pd.Timestamp.today().strftime("%Y-%m-%d")

    # Download stock data from Yahoo Finance for Example you can use any other source you use to work with
    df = yf.download(stock, start=date, interval=interval)


    ######USE YOUR OWN STRATEGY HERE######
    ######It could be anything from a simple moving average crossover to a complex deep learning model######


    # Define the action based on the prediction
    action= "BUY" if action == 1 else "SELL_SHORT"

    
    ##########PREVIEW and PLACE ORDER##########

    # #Calculate the quantity of shares to buy        
    quantity=int(amount//price)

    #generate a random orderid
    orderid=str(random.randint(10000000000,99999999999))

    #preview order
    payload = f"""<PreviewOrderRequest>
                <orderType>EQ</orderType>
                <clientOrderId>{orderid}</clientOrderId>
                <Order>
                    <allOrNone>false</allOrNone>
                    <priceType>MARKET</priceType>  
                    <orderTerm>GOOD_FOR_DAY</orderTerm>   
                    <marketSession>REGULAR</marketSession>
                    <stopPrice></stopPrice>
                    <limitPrice></limitPrice>
                    <Instrument>
                        <Product>
                            <securityType>EQ</securityType>
                            <symbol>{stock}</symbol>
                        </Product>
                        <orderAction>{action}</orderAction> 
                        <quantityType>QUANTITY</quantityType>
                        <quantity>{quantity}</quantity>
                    </Instrument>
                </Order>
            </PreviewOrderRequest>"""

    #send the request
    response=session.post(previewurl,header_auth=True,headers=headers,data=payload)
    logger.debug("Payload: %s", payload)
    logger.debug("Request Header: %s", response.request.headers)
    logger.debug("Response: %s", response.json())

    #get the previewid
    previewId_data = response.json().get('PreviewOrderResponse', {}).get('PreviewIds', [{}])[0]
    previewId = previewId_data.get('previewId', None)
 
    #place order
    orderid2=str(random.randint(100000000,99999999999))
    payloads= f"""<PlaceOrderRequest>
                    <orderType>EQ</orderType>
                    <clientOrderId>{orderid2}</clientOrderId>
                    <PreviewIds>
                        <previewId>{previewId}</previewId>
                    </PreviewIds>    
                        <Order>
                            <allOrNone>false</allOrNone>
                            <priceType>MARKET</priceType>
                            <orderTerm>GOOD_FOR_DAY</orderTerm>
                            <marketSession>REGULAR</marketSession>
                            <stopPrice />
                            <limitPrice />
                            <Instrument>
                                <Product>
                                    <securityType>EQ</securityType>
                                    <symbol>{stock}</symbol>
                                </Product>
                                <orderAction>{action}</orderAction>
                                <quantityType>QUANTITY</quantityType>
                                <quantity>{quantity}</quantity>
                            </Instrument>
                        </Order>
                    </PlaceOrderRequest>"""
                
    response=session.post(orderurl,header_auth=True,headers=headers,data=payloads)
    logger.debug("Request Header: %s", response.request.headers)
    logger.debug("Request Payload: %s", payloads)
    logger.debug("Response: %s", response.json())

    #print the order detail response in dataframes justs like account list
    Order_Placed=response['PlaceOrderResponse']['Order'][0]['estimatedTotalAmount']
    #print order placed
    print(Order_Placed)



    ############TAKE-PROFIT and STOP-LOSS ORDERS##########



    # Calculate limit prices for take-profit and stop-loss orders
    if action == 'BUY':
        take_profit_limit_price = round(price * (1+threshold), 2) 
        stop_loss_limit_price = round(price * (1-threshold), 2)
        order_action = 'SELL'
    else:
        stop_loss_limit_price = round(price * (1-threshold), 2)
        take_profit_limit_price = round(price * (1+threshold), 2)
        order_action = 'BUY_TO_COVER'



    #Create loop to check spot price  for take-profit and stop-loss orders
        
    while True:
    
        response=session.get(quoteurl, header_auth=True)
        logger.debug(response.json())
        lastprice=response.json()['QuoteResponse']['QuoteData'][0]['All']['lastTrade']
        
        if lastprice >= take_profit_limit_price or lastprice <= stop_loss_limit_price:
                print(f"Target reached. Preparing to place closing order. Last trade price: {lastprice}")
                # Generate another unique client order ID for the take-profit order
                orderid_tp = str(random.randint(10000000000, 99999999999))
                # Preview Take-profit limit order or Stop Loss
                closepreview = f"""<PreviewOrderRequest>
                            <orderType>EQ</orderType>
                            <clientOrderId>{orderid_tp}</clientOrderId>
                            <Order>
                                <allOrNone>false</allOrNone>
                                <priceType>MARKET</priceType>  
                                <orderTerm>GOOD_FOR_DAY</orderTerm>   
                                <marketSession>REGULAR</marketSession>
                                <stopPrice />
                                <limitPrice />
                                <Instrument>
                                    <Product>
                                        <securityType>EQ</securityType>
                                        <symbol>{stock}</symbol>
                                    </Product>
                                    <orderAction>{order_action}</orderAction> 
                                    <quantityType>QUANTITY</quantityType>
                                    <quantity>{quantity}</quantity>
                                </Instrument>
                            </Order>
                        </PreviewOrderRequest>"""
                response=session.post(previewurl,header_auth=True,headers=headers,data=closepreview)
                logger.debug("Request Header: %s", response.request.headers)
                logger.debug("Request Payload: %s", closepreview)
                logger.debug("Response: %s", response.json())
                #get the previewid
                previewId_data = response.json().get('PreviewOrderResponse', {}).get('PreviewIds', [{}])[0]
                previewId = previewId_data.get('previewId', None)
                
                #Place take_profit order or stop loss order
                orderid2_tp=str(random.randint(100000000,99999999999))
                closeplace= f"""<PlaceOrderRequest>
                                <orderType>EQ</orderType>
                                <clientOrderId>{orderid2_tp}</clientOrderId>
                                <PreviewIds>
                                    <previewId>{previewId}</previewId>
                                </PreviewIds>    
                                    <Order>
                                        <allOrNone>false</allOrNone>
                                        <priceType>MARKET</priceType>
                                        <orderTerm>GOOD_FOR_DAY</orderTerm>
                                        <marketSession>REGULAR</marketSession>
                                        <stopPrice />
                                        <limitPrice />
                                        <Instrument>
                                            <Product>
                                                <securityType>EQ</securityType>
                                                <symbol>{stock}</symbol>
                                            </Product>
                                            <orderAction>{order_action}</orderAction>
                                            <quantityType>QUANTITY</quantityType>
                                            <quantity>{quantity}</quantity>
                                        </Instrument>
                                    </Order>
                                </PlaceOrderRequest>"""
                response=session.post(orderurl,header_auth=True,headers=headers,data=closeplace)
                logger.debug("Request Header: %s", response.request.headers)
                logger.debug("Request Payload: %s", closeplace)
                logger.debug("Response: %s", response.json())
                
                 #print the order detail response in dataframes justs like account list
                Order_Closed=response['PlaceOrderResponse']['Order'][0]['estimatedTotalAmount']
                #print order placed
                print(Order_Closed)
                print(Order_Closed)

                #Print Profit last trade
                print(f"Profit: {Order_Closed-Order_Placed}")


                break
        time.sleep(0.1)




In [ ]:
def run_continuously(stock, amount):
    while True:
        try:
            quant(stock, amount)
            print(f"Quant function execution completed for {stock}. Restarting...")
             
        except KeyboardInterrupt:
            print(f"Process manually interrupted by the user for {stock}. Exiting...")
            break
        except Exception as e:
            print(f"An error occurred during quant execution for {stock}: {e}")



if __name__ == "__main__":

    ###### List of stock or stocks to run the quant function on ######

    stocks = ["","",""]
    threads = []

    # Create and start a thread for each stock
    for stock in stocks:
        thread = threading.Thread(target=run_continuously, args=(stock, 21000))
        threads.append(thread)
        thread.start()

    # Wait for all threads to complete
    for thread in threads:
        thread.join()

In [ ]:

headers = {"Content-Type": "application/xml", "consumerKey": CONSUMER_KEY}
logger = logging.getLogger('my_logger')
logger.setLevel(logging.DEBUG)
handler = RotatingFileHandler("python_client.log", maxBytes=5*1024*1024, backupCount=3)
FORMAT = "%(asctime)-15s %(message)s"
fmt = logging.Formatter(FORMAT, datefmt='%m/%d/%Y %I:%M:%S %p')
handler.setFormatter(fmt)
logger.addHandler(handler)

# function to get the stock data    
etrade=OAuth1Service(
    name="etrade",
    consumer_key=CONSUMER_KEY,
    consumer_secret=CONSUMER_SECRET,
    request_token_url="https://api.etrade.com/oauth/request_token",
    access_token_url="https://api.etrade.com/oauth/access_token",
    authorize_url="https://us.etrade.com/e/t/etws/authorize?key={}&token={}",
    base_url="https://api.etrade.com")

#Set the base url to the sandbox or production environment

#base_url=PROD_BASE_URL
#base_url=SANDBOX_BASE_URL

#Get the request token
request_token, request_token_secret = etrade.get_request_token(
    params={"oauth_callback": "oob", "format": "json"})
authorize_url=etrade.authorize_url.format(etrade.consumer_key, request_token)


#Open the browser to the authorize url to get the verifier c
webbrowser.open(authorize_url)

In [ ]:
#Get the access token with the verifier code
text_code="8Q0W2"
session = etrade.get_auth_session(request_token,
                                  request_token_secret,
                                  params={"oauth_verifier": text_code})
#print session status
print(session)


In [ ]:
#Get the account list
account_list=session.get(base_url+"/v1/accounts/list.json")
#get account list 
account_list.json()['AccountListResponse']['Accounts']['Account']
#clean them into data frame
account_list=pd.DataFrame(account_list.json()['AccountListResponse']['Accounts']['Account'])
account_list

In [ ]:
#Choose account to trade with and assign accountIdKey
accountIdKey=""
previewurl=base_url+"/v1/accounts/"+accountIdKey+"/orders/preview.json"
orderurl=base_url+"/v1/accounts/"+accountIdKey+"/orders/place.json"

In [ ]:
def quant2(stock,amount,threshold):

    symbol=stock

    #get current stock quote
    quoteurl=f"{base_url}/v1/market/quote/{stock}.json"
    response=session.get(quoteurl, header_auth=True)
    price=response.json()['QuoteResponse']['QuoteData'][0]['All']['lastTrade']

    #get today's date
    date = pd.Timestamp.today().strftime("%Y-%m-%d")

    # Download stock data from Yahoo Finance
    df = yf.download(stock, start=date, interval="1m")

    # Calculate Relative Strength Index (RSI)
    def calculate_rsi(series, period=10):
        delta = series.diff(1)
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        return 100 - (100 / (1 + rs))
    df['RSI'] = calculate_rsi(df['Adj Close'], period=10)

    # Calculate Moving Average Convergence Divergence (MACD)
    exp1 = df['Adj Close'].ewm(span=12, adjust=False).mean()
    exp2 = df['Adj Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = exp1 - exp2
    df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()
    # Calculate Volume Weighted Average Price (VWAP)
    df['VWAP'] = (df['Volume'] * (df['High'] + df['Low'] + df['Close']) / 3).cumsum() / df['Volume'].cumsum()
    df=df.dropna().copy()
    columns = ['Adj Close', 'Volume', 'RSI', 'MACD', 'Signal_Line', 'VWAP']
    df=df[columns]

    #time steps that the model will look back to make a prediction
    n_steps = 25

    # Define the target variable
    df['Target'] = (df['Adj Close'].shift(-1) > df['Adj Close']).astype(int)

    # Split the data
    split = int(0.8 * len(df))
    train_data = df[:split]
    test_data = df[split:]

    # Define the create_sequences function
    def create_sequences(X, y, n_steps):
        Xs, ys = [], []
        for i in range(len(X) - n_steps):
            Xs.append(X.iloc[i:(i+n_steps)].values)
            ys.append(y.iloc[i+n_steps])
        return np.array(Xs), np.array(ys)
    
    # Define the target variable
    train_targets = train_data['Target']
    test_targets = test_data['Target']

    # Create sequences
    X_train, y_train = create_sequences(train_data.drop(columns=['Target']), train_targets, n_steps)
    X_test, y_test = create_sequences(test_data.drop(columns=['Target']), test_targets, n_steps)

    # Normalize features separately for training and testing data
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
    X_test_scaled = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)
   
   # Define and compile the model
    model = Sequential([
        Input(shape=(X_train_scaled.shape[1], X_train_scaled.shape[2])),  # Input layer
        LSTM(50, activation='tanh', return_sequences=True),  # No input_shape here
        Dropout(0.2),
        LSTM(50, activation='tanh'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
   
    # Add callbacks
    early_stopping = EarlyStopping(monitor='val_loss',min_delta=1e-8, patience=10,verbose=0,mode="min",restore_best_weights=True)
    
    # Train the model
    history = model.fit(X_train_scaled, y_train, epochs=128, batch_size=30, validation_split=0.2, callbacks=early_stopping)

    # Predictions for the next step
    y_pred_next_step = (model.predict(X_test[-1].reshape(1, n_steps, X_test.shape[2])) > 0.5).astype("int32")
    action=y_pred_next_step[0][0]
    
    # Define the action based on the prediction
    action= "BUY" if action == 1 else "SELL_SHORT"

    #Print the current price and the predicted action
    print(f"Current price: {price}")
    print(f"Predicted action: {action}")
    
    print(f"Current price: {price}")
    print(f"Predicted action: {action}")
    

 
    #Calculate the quantity of shares to buy        
    quantity=int(amount//price)
    #generate a random orderid
    orderid=str(random.randint(10000000000,99999999999))
    #preview order
    payload = f"""<PreviewOrderRequest>
                <orderType>EQ</orderType>
                <clientOrderId>{orderid}</clientOrderId>
                <Order>
                    <allOrNone>false</allOrNone>
                    <priceType>MARKET</priceType>  
                    <orderTerm>GOOD_FOR_DAY</orderTerm>   
                    <marketSession>REGULAR</marketSession>
                    <stopPrice></stopPrice>
                    <limitPrice></limitPrice>
                    <Instrument>
                        <Product>
                            <securityType>EQ</securityType>
                            <symbol>{symbol}</symbol>
                        </Product>
                        <orderAction>{action}</orderAction> 
                        <quantityType>QUANTITY</quantityType>
                        <quantity>{quantity}</quantity>
                    </Instrument>
                </Order>
            </PreviewOrderRequest>"""

    #send the request
    response=session.post(previewurl,header_auth=True,headers=headers,data=payload)
    logger.debug("Payload: %s", payload)
    logger.debug("Request Header: %s", response.request.headers)
    logger.debug("Response: %s", response.json())

    #get the previewid
    previewId_data = response.json().get('PreviewOrderResponse', {}).get('PreviewIds', [{}])[0]
    previewId = previewId_data.get('previewId', None)
 
    #place order
    orderid2=str(random.randint(100000000,99999999999))
    payloads= f"""<PlaceOrderRequest>
                    <orderType>EQ</orderType>
                    <clientOrderId>{orderid2}</clientOrderId>
                    <PreviewIds>
                        <previewId>{previewId}</previewId>
                    </PreviewIds>    
                        <Order>
                            <allOrNone>false</allOrNone>
                            <priceType>MARKET</priceType>
                            <orderTerm>GOOD_FOR_DAY</orderTerm>
                            <marketSession>REGULAR</marketSession>
                            <stopPrice />
                            <limitPrice />
                            <Instrument>
                                <Product>
                                    <securityType>EQ</securityType>
                                    <symbol>{symbol}</symbol>
                                </Product>
                                <orderAction>{action}</orderAction>
                                <quantityType>QUANTITY</quantityType>
                                <quantity>{quantity}</quantity>
                            </Instrument>
                        </Order>
                    </PlaceOrderRequest>"""
                
    response=session.post(orderurl,header_auth=True,headers=headers,data=payloads)
    logger.debug("Request Header: %s", response.request.headers)
    logger.debug("Request Payload: %s", payloads)
    logger.debug("Response: %s", response.json())

    #print the order detail response in dataframes justs like account list
    Order_Placed=response['PlaceOrderResponse']['Order'][0]['estimatedTotalAmount']
    #print order placed
    print(Order_Placed)
    print(Order_Placed)
    
    

    # Calculate limit prices for take-profit and stop-loss orders
    if action == 'BUY':
        take_profit_limit_price = round(price * (1+threshold), 2) 
        stop_loss_limit_price = round(price * (1-threshold), 2)
        order_action = 'SELL'
    else:
        stop_loss_limit_price = round(price * (1-threshold), 2)
        take_profit_limit_price = round(price * (1+threshold), 2)
        order_action = 'BUY_TO_COVER'
    #Create loop to check spot price  for take-profit and stop-loss orders
        
    while True:
    
        response=session.get(quoteurl, header_auth=True)
        logger.debug(response.json())
        lastprice=response.json()['QuoteResponse']['QuoteData'][0]['All']['lastTrade']
        
        if lastprice >= take_profit_limit_price or lastprice <= stop_loss_limit_price:
                print(f"Target reached. Preparing to place closing order. Last trade price: {lastprice}")
                # Generate another unique client order ID for the take-profit order
                orderid_tp = str(random.randint(10000000000, 99999999999))
                # Preview Take-profit limit order or Stop Loss
                closepreview = f"""<PreviewOrderRequest>
                            <orderType>EQ</orderType>
                            <clientOrderId>{orderid_tp}</clientOrderId>
                            <Order>
                                <allOrNone>false</allOrNone>
                                <priceType>MARKET</priceType>  
                                <orderTerm>GOOD_FOR_DAY</orderTerm>   
                                <marketSession>REGULAR</marketSession>
                                <stopPrice />
                                <limitPrice />
                                <Instrument>
                                    <Product>
                                        <securityType>EQ</securityType>
                                        <symbol>{symbol}</symbol>
                                    </Product>
                                    <orderAction>{order_action}</orderAction> 
                                    <quantityType>QUANTITY</quantityType>
                                    <quantity>{quantity}</quantity>
                                </Instrument>
                            </Order>
                        </PreviewOrderRequest>"""
                response=session.post(previewurl,header_auth=True,headers=headers,data=closepreview)
                logger.debug("Request Header: %s", response.request.headers)
                logger.debug("Request Payload: %s", closepreview)
                logger.debug("Response: %s", response.json())
                #get the previewid
                previewId_data = response.json().get('PreviewOrderResponse', {}).get('PreviewIds', [{}])[0]
                previewId = previewId_data.get('previewId', None)
                
                #Place take_profit order or stop loss order
                orderid2_tp=str(random.randint(100000000,99999999999))
                closeplace= f"""<PlaceOrderRequest>
                                <orderType>EQ</orderType>
                                <clientOrderId>{orderid2_tp}</clientOrderId>
                                <PreviewIds>
                                    <previewId>{previewId}</previewId>
                                </PreviewIds>    
                                    <Order>
                                        <allOrNone>false</allOrNone>
                                        <priceType>MARKET</priceType>
                                        <orderTerm>GOOD_FOR_DAY</orderTerm>
                                        <marketSession>REGULAR</marketSession>
                                        <stopPrice />
                                        <limitPrice />
                                        <Instrument>
                                            <Product>
                                                <securityType>EQ</securityType>
                                                <symbol>{symbol}</symbol>
                                            </Product>
                                            <orderAction>{order_action}</orderAction>
                                            <quantityType>QUANTITY</quantityType>
                                            <quantity>{quantity}</quantity>
                                        </Instrument>
                                    </Order>
                                </PlaceOrderRequest>"""
                response=session.post(orderurl,header_auth=True,headers=headers,data=closeplace)
                logger.debug("Request Header: %s", response.request.headers)
                logger.debug("Request Payload: %s", closeplace)
                logger.debug("Response: %s", response.json())
                
                 #print the order detail response in dataframes justs like account list
                Order_Closed=response['PlaceOrderResponse']['Order'][0]['estimatedTotalAmount']
                #print order placed
                print(Order_Closed)
                print(Order_Closed)

                #Print Profit last trade
                print(f"Profit: {Order_Closed-Order_Placed}")
                print(f"Profit: {Order_Closed-Order_Placed}")



                break
        time.sleep(0.1)
        

In [ ]:
def run_continuously(stock, amount):
    while True:
        try:
            quant2(stock, amount)
            print(f"Quant function execution completed for {stock}. Restarting...")
             
        except KeyboardInterrupt:
            print(f"Process manually interrupted by the user for {stock}. Exiting...")
            break
        except Exception as e:
            print(f"An error occurred during quant execution for {stock}: {e}")


if __name__ == "__main__":

    ###### List of stock or stocks to run the quant function on ######

    stocks = ["","",""]
    threads = []

    # Create and start a thread for each stock
    for stock in stocks:
        thread = threading.Thread(target=run_continuously, args=(stock, 21000))
        threads.append(thread)
        thread.start()

    # Wait for all threads to complete
    for thread in threads:
        thread.join()